# 1. Sen4AgriNet labels and parcel locations, then annual Sentinel-2 imagery

**Run All is configured.** Scan all years and tiles in one pinned release, reading remote reference masks through bounded HTTP byte ranges. Full NetCDF imagery files are not downloaded. Extract eligible carrot, cabbage and potato parcel locations, then collect their annual Sentinel-2 imagery separately through Earth Engine.

Existing raw files, reference inventories and analytical results remain unchanged. New reference results use a separate versioned folder. Opening the notebook does not execute retrieval.

**Progress and time estimates:** the top bar counts completed stages; inner bars count finished patches, parcels, files, pixels or models. Stage ETA uses observed speed. Whole-notebook ETA uses a comparable completed run, or measured remaining work in the final stage; otherwise it says `learning`. Estimates are rough and include network variability. Long operations advance when their work unit finishes. The final cell prints elapsed time and the saved timing-report path.


## 1. Fresh-clone setup

Choose a Python 3.12+ notebook kernel (3.13 recommended), then **Run All**. The first cell makes the cloned package importable, installs missing dependencies in that kernel, and creates `settings.local.json` with portable local paths if absent. Existing settings are preserved.

If no Earth Engine project is configured, enter your registered project ID once when prompted. It is saved locally. Complete the later Google sign-in prompt. No saved NetCDFs, inventories or imagery are required; subsequent stages obtain their inputs automatically.

**Output:** kernel setup, local folders and project configuration. Full-study reference and satellite collection switches are enabled for Run All.


In [ ]:
# Standard-library setup runs before scientific imports, so a fresh clone needs no saved data.
from pathlib import Path
import sys

notebook_repository_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
notebook_source_folder = str(notebook_repository_root / "src")
if notebook_source_folder not in sys.path:
    sys.path.insert(0, notebook_source_folder)
from crop_pipeline import notebook_setup
notebook_setup.prepare_environment(notebook_repository_root)

from pathlib import Path
import json
import os
from crop_pipeline.sources import annual
from crop_pipeline.sources import sen4agrinet as sen4agrinet_references

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
settings_path = Path("settings.local.json")
local_settings = notebook_setup.ensure_local_settings(settings_path)
report_folder = Path(local_settings.get("report_root", "./sen4agrinet_work/inventory")).expanduser().resolve()
dataset_folder = Path(os.environ.get("S4A_ROOT", local_settings.get("dataset_root", "./ground_truth"))).expanduser().resolve()
annual_archive_root = annual.archive_root(local_settings, report_folder)
reference_root = Path(local_settings.get("sen4agrinet_reference_root", str(annual_archive_root / "sen4agrinet_references"))).expanduser().resolve()
run_name = annual.COLLECTION_NAME
earth_engine_project = os.environ.get("EE_PROJECT", local_settings.get("earth_engine_project", ""))

# Run All must finish the full reference inventory before starting satellite collection.
retrieve_reference_masks = True
rebuild_field_inventory = True
run_satellite_collection = True
authenticate_now = True

# None reuses the first pinned release. Set a specific release SHA only to start another release.
dataset_revision = None
print("Existing raw files, read-only reuse:", dataset_folder)
print("Versioned full-reference outputs:", reference_root)
print("Local satellite archive:", annual_archive_root)

# Resolve account setup before large reference transfers start.
earth_engine_project = notebook_setup.project_for_collection(local_settings, settings_path, required=run_satellite_collection)

from crop_pipeline import progress as pipeline_progress
notebook_progress = pipeline_progress.start_notebook(
    "01_sen4agrinet_preparation",
    ['Certificates', 'Discover reference patches', 'Extract reference masks', 'Prepare eligible parcels', 'Collect annual imagery'],
    report_folder.parent / "progress",
    workload={"folder": str(annual_archive_root), "satellite_collection": globals().get("run_satellite_collection"), "training_mode": globals().get("training_mode")},
)


### HTTPS certificates

The bootstrap configures trust before the first request. This cell displays the trusted-certificate count before reference discovery; HTTPS verification remains enabled.


In [ ]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Certificates')

import ssl
import certifi
os.environ.setdefault("SSL_CERT_FILE", certifi.where())
print("Trusted certificates:", ssl.create_default_context().cert_store_stats()["x509_ca"])

pipeline_progress.end_stage("completed")


## 2. Discover every year and tile

List every NetCDF patch under the pinned repository's `data/` tree. There is no year, tile or candidate-budget filter. The same pinned release is reused across restarts.

**Output:** release SHA, patch count, years and year/tile combinations. The displayed full NetCDF size is context only: it is **not** the planned transfer. Reference-only transfer size depends on mask compression and HDF5 layout and cannot be known exactly before reading. The next stage reports actual received bytes.


In [ ]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Discover reference patches')

repository_manifest = sen4agrinet_references.discover_release(
    reference_root, enabled=retrieve_reference_masks, revision=dataset_revision,
)
reference_run_folder = sen4agrinet_references.run_folder(reference_root, repository_manifest)
print("Pinned dataset release:", repository_manifest["revision"])
print("NetCDF patches:", len(repository_manifest["files"]))
print("Years:", repository_manifest["years"])
print("Year/tile combinations:", len(repository_manifest["year_tile_pairs"]))
print("Full NetCDF size, NOT the planned transfer (GiB):", round(repository_manifest["total_source_bytes"] / 1024**3, 2))
print("Largest upstream NetCDF, NOT downloaded whole (GiB):", round(max(record["expected_bytes"] for record in repository_manifest["files"]) / 1024**3, 2))
print("Reference run:", reference_run_folder)

pipeline_progress.end_stage("completed")


## 3. Read reference masks without downloading full NetCDF files

Reuse verified local files when available. Otherwise, request small byte ranges for HDF5 metadata, the label mask and its coordinates. Read the parcel mask only when target labels are present. Spectral arrays are not accessed. A small amount of adjacent data may share requested byte blocks.

Save compressed mask-only NetCDF caches and extracted-reference checkpoints locally. Complete checkpoints avoid further reads; mask caches can survive feature-extractor changes. No complete imagery NetCDF is staged or deleted in this mode. If a server ignores HTTP Range, stop before reading its body; there is **no automatic full-file fallback**.

**Eligibility stays unchanged:** crop codes, valid masks, 10 m grid/coordinate checks, parcel IDs, geometry validation and ambiguous-group exclusions. Every patch must still be scanned; any failure blocks the completed reference handoff.

**Output:** progress per patch, eligible crop counts, excluded groups, actual received reference MiB and checkpoint reuse count. The source is pinned by release/path and checked HTTP offsets/size. Partial reads cannot verify the entire upstream NetCDF checksum; downloaded mask caches have their own SHA-256 integrity checks.


In [ ]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Extract reference masks')

if rebuild_field_inventory:
    reference_summary = sen4agrinet_references.build_reference_inventory(
        repository_manifest, reference_root, dataset_folder,
        download_enabled=retrieve_reference_masks,
        reference_only=True,
    )
else:
    reference_summary = sen4agrinet_references.load_completed_inventory(reference_run_folder, repository_manifest)
print("Reference scan:", reference_summary["status"])
print("Files scanned:", reference_summary["files_scanned"])
print("Eligible crop counts:", reference_summary["crop_counts"])
print("Excluded ambiguous parcel groups:", reference_summary["excluded_parcel_groups"])
print("Full-release fields:", reference_summary["field_root"])

transfer_path = reference_run_folder / "last_scan_transfer.json"
if transfer_path.exists():
    transfer = json.loads(transfer_path.read_text())
    print("Reference bytes received this scan (MiB):", round(transfer["network_bytes"] / 1024**2, 2))
    print("Patches reused from checkpoints:", transfer["patches_reused_from_checkpoints"])

pipeline_progress.end_stage("completed")


## 4. Prepare all eligible parcel pixels and estimate satellite storage

Read only the newly completed full-release inventory—not the old pilot subset. Merge same-parcel fragments, remove competing spatial claims and retain all eligible crop-mask pixels. Large footprints use bounded tiles while preserving one parcel group for training splits.

**Output:** parcel counts, tile counts, 24-bin image-file counts and uncompressed satellite raster GiB. This estimate is separate from the NetCDF transfer shown above. No per-crop or per-parcel pilot quota is applied.


In [ ]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Prepare eligible parcels')

reference_features, inventory_provenance = annual.load_sen4agrinet_references(Path(reference_summary["field_root"]))
annual_plan = annual.prepare_source(
    "sen4agrinet", reference_features, annual_archive_root,
    provenance={"dataset": sen4agrinet_references.DATASET_NAME,
                "dataset_revision": repository_manifest["revision"],
                "coverage": "all_patches_all_years_all_tiles_in_pinned_release",
                "repository_files_scanned": reference_summary["files_scanned"],
                "reference_summary": str(reference_run_folder / "completed_inventory.json"),
                "inventories": inventory_provenance},
    inward_buffer_metres=0,
)
print("Satellite estimate:", annual_plan["estimate"])
print("Reference years:", list(annual_plan["years"]))

pipeline_progress.set_workload({"folder": str(annual_archive_root), "source_revision": annual_plan["revision"], "satellite_collection": run_satellite_collection})

pipeline_progress.end_stage("completed")


## 5. Collect full-year Sentinel-2 imagery

Collection and sign-in are enabled for Run All. Earth Engine applies the existing cloud masks, daily mosaics, bilinear B5/B11 alignment and half-month median recipe. Save all 24 bins in every reference year, together with dated vegetation summaries, to the local archive. Window detection does not filter downloads here.

**Output:** `status: completed` means every eligible planned tile has all annual files and daily-summary checkpoints. Cloudy files can still fail later quality checks. Existing verified satellite files are reused on restart.


In [ ]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Collect annual imagery')

collection_result = annual.collect_source(
    annual_plan, report_folder, earth_engine_project,
    enabled=run_satellite_collection, authenticate=authenticate_now,
)
print(collection_result)

pipeline_progress.end_stage("completed" if run_satellite_collection else "skipped")
pipeline_progress.finish_notebook()


## 6. Continue after all source collections finish

Finish [FranceCrops](02_francecrops_preparation.ipynb) and [CropHarvest](03_cropharvest_preparation.ipynb), then run [04 Four-month windows](04_four_month_windows.ipynb), [05 Feature engineering](05_feature_engineering.ipynb), and [06 Model training](06_model_training.ipynb).

Notebook 04 requires all three completed handoffs. It applies timing and quality checks to the saved annual archive and chooses the earliest complete usable window per parcel. Old results remain in their original folders. January–December archives still require an explicit adjacent-year extension for boundary-crossing windows.
